In [1]:
# ==========================================
# THESIS DATA ANALYSIS: COMPLETE AUTOMATION
# Logic: Shapiro-Wilk -> (Normal?) -> Paired T-Test OR Wilcoxon
# Includes: Detailed Tables & Formula Walkthroughs for ALL tests.
# ==========================================

import numpy as np
import scipy.stats as stats
import math
import pandas as pd

# Coefficients for Shapiro-Wilk (Approximated for n=3 to 10)
SHAPIRO_COEFFS = {
    3: [0.7071],
    4: [0.6872, 0.1677],
    5: [0.6646, 0.2413],
    6: [0.6431, 0.2806, 0.0875],
    7: [0.6233, 0.3031, 0.1401],
    8: [0.6052, 0.3164, 0.1743, 0.0561],
    9: [0.5888, 0.3244, 0.1976, 0.0947],
    10: [0.5739, 0.3291, 0.2141, 0.1224, 0.0399]
}

def get_user_input(scenario, model_name):
    print(f"\n--- [{scenario}] Enter Accuracy Scores for {model_name} ---")
    print("Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)")
    input_str = input(f"Scores for {model_name}: ")
    try:
        scores = [float(x.strip()) for x in input_str.split(',')]
        return np.array(scores)
    except ValueError:
        print("Error: Invalid input. Please try again.")
        return get_user_input(scenario, model_name)

# ==========================================
# TEST 1: SHAPIRO-WILK (Normality)
# ==========================================
def explain_shapiro_wilk(data, name):
    n = len(data)
    print(f"\n\n{'='*80}")
    print(f"NORMALITY CHECK: {name} (n={n})")
    print(f"{'='*80}")
    
    if n < 3:
        print("Sample size too small for formula walkthrough (n must be >= 3).")
        return False # Treat as non-normal for safety
    
    # 1. Basic Stats
    sorted_data = np.sort(data)
    mean_val = np.mean(data)
    
    # 2. Prepare Columns
    col_xi = sorted_data
    col_sq_diff = [(x - mean_val)**2 for x in sorted_data]
    col_ai = [""] * n
    col_term = [""] * n
    
    b_value = 0
    
    if n in SHAPIRO_COEFFS:
        coeffs = SHAPIRO_COEFFS[n]
        k = len(coeffs)
        
        for i in range(k):
            coeff = coeffs[i]
            upper = sorted_data[-(i+1)]
            lower = sorted_data[i]
            diff = upper - lower
            term = coeff * diff
            
            col_ai[i] = coeff
            col_term[i] = term 
            b_value += term
    
    # 3. Construct DataFrame
    df_shapiro = pd.DataFrame({
        'Index': range(1, n + 1),
        'Accuracy (xi)': col_xi,
        '(xi - x̄)²': col_sq_diff,
        'ai': col_ai,
        'Term (ai * Diff)': col_term
    })
    
    pd.options.display.float_format = '{:.4f}'.format
    print(f"\n[Calculation Table]")
    print("-" * 85)
    print(df_shapiro.to_string(index=False))
    print("-" * 85)
    
    # 4. Final Sums
    SS = sum(col_sq_diff)
    W_calc = (b_value ** 2) / SS if SS != 0 else 0
    
    print(f"   Mean (x̄) = {mean_val:.4f}")
    print(f"   Sum of Squares (SS) = {SS:.5f}")
    print(f"   b (Sum of Terms) = {b_value:.5f}")
    print(f"   W = b² / SS = ({b_value:.4f})² / {SS:.4f} = {W_calc:.4f}")
    
    # Verification & Decision
    stat, p_val = stats.shapiro(data)
    is_normal = p_val > 0.05
    decision = "Normal (Fail to Reject Ho)" if is_normal else "Not Normal (Reject Ho)"
    print(f"\n   [Verification] SciPy p-value = {p_val:.4f} -> {decision}")
    
    return is_normal

# ==========================================
# TEST 2: PAIRED T-TEST (Parametric)
# ==========================================
def perform_detailed_paired_ttest(group1, group2, name1, name2, scenario):
    """
    Calculates Paired T-Test with formula walkthrough.
    Used when data IS normally distributed.
    """
    print(f"\n\n{'='*80}")
    print(f"PARAMETRIC TEST: Paired T-Test ({name1} vs {name2})")
    print(f"Condition: {scenario} (Data is Normal)")
    print(f"{'='*80}")
    
    n = len(group1)
    
    # 1. Calculate Differences
    differences = group1 - group2
    sq_diffs = differences ** 2
    
    # 2. Data Table
    df_ttest = pd.DataFrame({
        f'{name1} (x1)': group1,
        f'{name2} (x2)': group2,
        'Difference (d)': differences,
        'Diff Squared (d²)': sq_diffs
    })
    
    print(f"\n[Step 1] Difference Calculation Table")
    print("-" * 60)
    print(df_ttest.to_string(index=False))
    print("-" * 60)
    
    # 3. Calculate Statistics
    sum_d = np.sum(differences)
    mean_d = np.mean(differences)
    sum_sq_diff_mean = np.sum((differences - mean_d)**2)
    
    # Standard Deviation of differences (Sample)
    sd = math.sqrt(sum_sq_diff_mean / (n - 1))
    
    # Standard Error
    se = sd / math.sqrt(n)
    
    # T-Statistic
    t_stat = mean_d / se
    
    print(f"\n[Step 2] Calculate Mean Difference & Std Dev")
    print(f"   Sum of differences (Σd) = {sum_d:.4f}")
    print(f"   Mean difference (d̄) = {sum_d:.4f} / {n} = {mean_d:.4f}")
    print(f"   Sum of (d - d̄)² = {sum_sq_diff_mean:.4f}")
    print(f"   Std Dev (sd) = sqrt({sum_sq_diff_mean:.4f} / {n-1}) = {sd:.4f}")
    
    print(f"\n[Step 3] Calculate T-Statistic")
    print(f"   Standard Error (SE) = sd / sqrt(n) = {sd:.4f} / {math.sqrt(n):.4f} = {se:.4f}")
    print(f"   t = d̄ / SE")
    print(f"   t = {mean_d:.4f} / {se:.4f} = {t_stat:.4f}")
    
    # 4. P-Value
    # Two-tailed p-value
    df = n - 1
    p_val = stats.t.sf(np.abs(t_stat), df) * 2
    decision = "Reject Ho" if p_val <= 0.05 else "Fail to Reject Ho"
    
    print(f"\n[Step 4] Final Results")
    print("-" * 50)
    print(f"{'Parameter':<15} | {'Value':<15}")
    print("-" * 50)
    print(f"{'t-value':<15} | {t_stat:<15.4f}")
    print(f"{'Degrees Freedom':<15} | {df:<15}")
    print(f"{'p-value':<15} | {p_val:<15.4f}")
    print(f"{'Decision':<15} | {decision:<15}")
    print("-" * 50)

# ==========================================
# TEST 3: WILCOXON RANK SUM (Non-Parametric)
# ==========================================
def perform_detailed_wilcoxon(group1, group2, name1, name2, scenario):
    """
    Calculates Wilcoxon Rank Sum (Mann-Whitney) with formula walkthrough.
    Used when data is NOT normally distributed.
    """
    print(f"\n\n{'='*80}")
    print(f"NON-PARAMETRIC TEST: Wilcoxon Rank Sum ({name1} vs {name2})")
    print(f"Condition: {scenario} (Data is NOT Normal)")
    print(f"{'='*80}")
    
    n1 = len(group1)
    n2 = len(group2)
    
    # --- STEP 1: RANKING ---
    combined_data = np.concatenate([group1, group2])
    labels = [name1] * n1 + [name2] * n2
    ranks = stats.rankdata(combined_data)
    
    df_ranks = pd.DataFrame({'Score': combined_data, 'Group': labels, 'Rank': ranks})
    df_ranks = df_ranks.sort_values(by='Score').reset_index(drop=True)
    
    print(f"\n[Step 1] Rank Assignment Table")
    print("-" * 40)
    print(df_ranks.to_string(index=False))
    print("-" * 40)
    
    # --- STEP 2: CALCULATE R ---
    r1_ranks = ranks[:n1] 
    R = np.sum(r1_ranks)
    
    print(f"\n[Step 2] Calculate Sum of Ranks (R)")
    print(f"   Sum ranks for {name1} (n1={n1}): {list(r1_ranks)}")
    print(f"   R = {R}")

    # --- STEP 3: THESIS FORMULAS ---
    mu_R = (n1 * (n1 + n2 + 1)) / 2
    sigma_R = math.sqrt((n1 * n2 * (n1 + n2 + 1)) / 12)
    
    if sigma_R == 0:
        print("Error: Standard deviation is 0.")
        return

    z_score = (R - mu_R) / sigma_R
    p_value_z = 2 * (1 - stats.norm.cdf(abs(z_score)))
    decision = "Reject Ho" if p_value_z <= 0.05 else "Fail to Reject Ho"
    
    print(f"\n[Step 3] Apply Formulas")
    print(f"   μR = {n1}({n1} + {n2} + 1) / 2  = {mu_R:.4f}")
    print(f"   σR = sqrt({n1}*{n2}*({n1 + n2 + 1}) / 12) = {sigma_R:.4f}")
    print(f"   z  = ({R} - {mu_R}) / {sigma_R:.4f} = {z_score:.4f}")
    
    print(f"\n[Step 4] Final Results")
    print("-" * 50)
    print(f"{'Parameter':<15} | {'Value':<15}")
    print("-" * 50)
    print(f"{'z-value':<15} | {z_score:<15.4f}")
    print(f"{'p-value':<15} | {p_value_z:<15.4f}")
    print(f"{'Decision':<15} | {decision:<15}")
    print("-" * 50)

# ==========================================
# MAIN EXECUTION LOGIC
# ==========================================
def analyze_comparison(group_exp, group_ctrl, name_exp, name_ctrl, scenario):
    # 1. Check Normality for BOTH groups
    norm_exp = explain_shapiro_wilk(group_exp, f"{name_exp} ({scenario})")
    norm_ctrl = explain_shapiro_wilk(group_ctrl, f"{name_ctrl} ({scenario})")
    
    # 2. Decide Test
    # If BOTH are Normal -> Paired T-Test
    # If EITHER is Not Normal -> Wilcoxon Rank Sum
    if norm_exp and norm_ctrl:
        perform_detailed_paired_ttest(group_exp, group_ctrl, name_exp, name_ctrl, scenario)
    else:
        perform_detailed_wilcoxon(group_exp, group_ctrl, name_exp, name_ctrl, scenario)

print("--- DATA ENTRY ---")

# Get Augmented Data
print("\n[SCENARIO 1] WITH AUGMENTATION")
aug_base = get_user_input("AUGMENTED", "Baseline Model")
aug_plsr = get_user_input("AUGMENTED", "Hybrid PLSR Model")
aug_xgb  = get_user_input("AUGMENTED", "Hybrid XGBoost Model")

# Get Non-Augmented Data
print("\n[SCENARIO 2] WITHOUT AUGMENTATION")
no_aug_base = get_user_input("NO AUGMENTATION", "Baseline Model")
no_aug_plsr = get_user_input("NO AUGMENTATION", "Hybrid PLSR Model")
no_aug_xgb  = get_user_input("NO AUGMENTATION", "Hybrid XGBoost Model")

# --- RUN ANALYSIS ---
print("\n" + "#"*60)
print("STARTING ANALYSIS: SCENARIO 1 (WITH AUGMENTATION)")
print("#"*60)
analyze_comparison(aug_plsr, aug_base, "Hybrid PLSR", "Baseline", "Augmented")
analyze_comparison(aug_xgb, aug_base, "Hybrid XGBoost", "Baseline", "Augmented")

print("\n" + "#"*60)
print("STARTING ANALYSIS: SCENARIO 2 (WITHOUT AUGMENTATION)")
print("#"*60)
analyze_comparison(no_aug_plsr, no_aug_base, "Hybrid PLSR", "Baseline", "Not Augmented")
analyze_comparison(no_aug_xgb, no_aug_base, "Hybrid XGBoost", "Baseline", "Not Augmented")

--- DATA ENTRY ---

[SCENARIO 1] WITH AUGMENTATION

--- [AUGMENTED] Enter Accuracy Scores for Baseline Model ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)


Scores for Baseline Model:  0.7644, 0.3942, 0.5472, 0.5590



--- [AUGMENTED] Enter Accuracy Scores for Hybrid PLSR Model ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)


Scores for Hybrid PLSR Model:  0.5409, 0.4904, 0.4505, 0.4599



--- [AUGMENTED] Enter Accuracy Scores for Hybrid XGBoost Model ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)


...


--- [NO AUGMENTATION] Enter Accuracy Scores for Hybrid PLSR Model ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)


Scores for Hybrid PLSR Model:  0.4087, 0.4976, 0.4670, 0.5142



--- [NO AUGMENTATION] Enter Accuracy Scores for Hybrid XGBoost Model ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)


Scores for Hybrid XGBoost Model:  0.4976, 0.4471, 0.4292, 0.4788



############################################################
STARTING ANALYSIS: SCENARIO 1 (WITH AUGMENTATION)
############################################################


NORMALITY CHECK: Hybrid PLSR (Augmented) (n=4)

[Calculation Table]
-------------------------------------------------------------------------------------
 Index  Accuracy (xi)  (xi - x̄)²     ai Term (ai * Diff)
     1         0.4505      0.0012 0.6872           0.0621
     2         0.4599      0.0007 0.1677           0.0051
     3         0.4904      0.0000                        
     4         0.5409      0.0031                        
-------------------------------------------------------------------------------------
   Mean (x̄) = 0.4854
   Sum of Squares (SS) = 0.00497
   b (Sum of Terms) = 0.06724
   W = b² / SS = (0.0672)² / 0.0050 = 0.9090

   [Verification] SciPy p-value = 0.4720 -> Normal (Fail to Reject Ho)


NORMALITY CHECK: Baseline (Augmented) (n=4)

[Calculation Table]
--------------------------

=== PART 1: WITH AUGMENTATION DATA ===

--- [WITH AUGMENTATION] Enter Accuracy Scores for Baseline ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)
Scores for Baseline:  0.7644, 0.3942, 0.5472, 0.5590

--- [WITH AUGMENTATION] Enter Accuracy Scores for Hybrid PLSR ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)
Scores for Hybrid PLSR:  0.5409, 0.4904, 0.4505, 0.4599

--- [WITH AUGMENTATION] Enter Accuracy Scores for Hybrid XGBoost ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)
Scores for Hybrid XGBoost:  0.5505, 0.4639, 0.4528, 0.4245

=== PART 2: WITHOUT AUGMENTATION DATA ===

--- [NO AUGMENTATION] Enter Accuracy Scores for Baseline ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)
Scores for Baseline:  0.5240, 0.4183, 0.6085, 0.6392

--- [NO AUGMENTATION] Enter Accuracy Scores for Hybrid PLSR ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)
Scores for Hybrid PLSR:  0.4087, 0.4976, 0.4670, 0.5142

--- [NO AUGMENTATION] Enter Accuracy Scores for Hybrid XGBoost ---
Enter scores separated by commas (e.g., 0.75, 0.80, 0.82)
Scores for Hybrid XGBoost:  0.4976, 0.4471, 0.4292, 0.4788
